In [18]:
import pandas as pd
import torch
import pickle
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

from transformers import AutoTokenizer, AutoModel

In [20]:
import pandas as pd

# 1. Load the CSV (either from local file or raw GitHub URL)
#    If you want to fetch directly, use the raw URL below; 
#    otherwise replace with 'harmonized-system.csv'.
url = 'https://raw.githubusercontent.com/datasets/harmonized-system/main/data/harmonized-system.csv'
df = pd.read_csv(url, dtype=str)


# get all of the rows that have level 2
chapters = df[df['level'] == str(2)]
headings = df[df['level'] == str(4)]
subheadings = df[df['level'] == str(6)]


chapters


,section,hscode,description,parent,level
0,I,01,Animals; live,TOTAL,2
41,I,02,Meat and edible meat offal,TOTAL,2
118,I,03,"Fish and crustaceans, molluscs and other aquat...",TOTAL,2
353,I,04,Dairy produce; birds' eggs; natural honey; edi...,TOTAL,2
397,I,05,Animal originated products; not elsewhere spec...,TOTAL,2
...,...,...,...,...,...
6733,XX,94,"Furniture; bedding, mattresses, mattress suppo...",TOTAL,2
6794,XX,95,"Toys, games and sports requisites; parts and a...",TOTAL,2
6840,XX,96,Miscellaneous manufactured articles,TOTAL,2
6910,XXI,97,Works of art; collectors' pieces and antiques,TOTAL,2


In [13]:
all_tariffs = pd.read_csv('all_tariffs.csv')

# get 2023 data

all_tariffs = all_tariffs[all_tariffs['year'] == 2023]

all_tariffs

/var/folders/87/vb506yp95bn1dt3x5p9sbk4r0000gn/T/ipykernel_64090/1098268842.py:1: DtypeWarning: Columns (1,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,26) have mixed types. Specify dtype option on import or set low_memory=False.
  all_tariffs = pd.read_csv('all_tariffs.csv')


,year,hs,brief_description,mfn_ad_val_rate,mfn_specific_rate,unit_1,unit_2,quantity_1_code,quantity_2_code,description_1,...,country_1,country_duty_1,country_2,country_duty_2,country_3,country_duty_3,country_desc,country_codes,mfn_other_rate,mfn_rate_type_code
745524,2023,1012100,Live purebred breeding horses,0.000,0.0,NaN,NaN,NO,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
745525,2023,1012900,Live horses other than purebred breeding horses,0.000,0.0,NaN,NaN,NO,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
745526,2023,1013000,Live asses,0.068,0.0,NaN,NaN,NO,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,7
745527,2023,1019030,Mules and hinnies imported for immediate slaug...,0.000,0.0,NaN,NaN,NO,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0
745528,2023,1019040,Mules and hinnies not imported for immediate s...,0.045,0.0,NaN,NaN,NO,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
758605,2023,99990061,Imports of wool apparel from Mexico described ...,NaN,NaN,NaN,NaN,SME,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
758606,2023,99990062,Imports of textile and apparel goods from Mexi...,NaN,NaN,NaN,NaN,SME,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
758607,2023,99990064,Imports of textile and apparel goods from Mexi...,NaN,NaN,NaN,NaN,SME,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N
758608,2023,99990084,Goods imported from Singapore and treated as o...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N


In [4]:
def setup_model_and_tokenizer():
    token = 'REDACTED_HF_TOKEN'
    model_name = 'sentence-transformers/all-mpnet-base-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
    model = AutoModel.from_pretrained(model_name, token=token)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    model.eval()
    return tokenizer, model, device

In [5]:
def encode_text(texts, tokenizer, model, device, batch_size=64):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encoded_input = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
        encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
        with torch.no_grad():
            model_output = model(**encoded_input)
        token_embeddings = model_output.last_hidden_state
        attention_mask = encoded_input['attention_mask']
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        embeddings = sum_embeddings / sum_mask
        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu())
    return torch.cat(all_embeddings, dim=0)

In [6]:
def process_year(year, df, tokenizer, model, device, output_dir):
    df_year = df[df['year'] == year]
    descriptions = df_year['brief_description'].tolist()
    if not descriptions:
        return None
    embeddings = encode_text(descriptions, tokenizer, model, device)
    embeddings_np = embeddings.cpu().numpy()
    out_path = os.path.join(output_dir, f'embeddings_{year}.pkl')
    with open(out_path, 'wb') as f:
        pickle.dump(embeddings_np, f)
    return out_path

In [7]:
df = pd.read_csv('all_tariffs.csv')
df = df.dropna(subset=['brief_description']).copy()
df['brief_description'] = df['brief_description'].astype(str)
years = sorted(df['year'].unique())
tokenizer, model, device = setup_model_and_tokenizer()
output_dir = '.'





/var/folders/87/vb506yp95bn1dt3x5p9sbk4r0000gn/T/ipykernel_49770/3295271234.py:1: DtypeWarning: Columns (1,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('all_tariffs.csv')


In [8]:
max_workers = min(32, (os.cpu_count() or 1) + 4)  # Dynamically set max_workers

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {}
    for year in years:
        out_path = os.path.join(output_dir, f'embeddings_{year}.pkl')
        if os.path.exists(out_path):
            print(f"Already exists: {out_path}, skipping.")
            continue
        futures[executor.submit(process_year, year, df, tokenizer, model, device, output_dir)] = year

    for future in tqdm(as_completed(futures), total=len(futures), desc="Processing years"):
        year = futures[future]
        try:
            out_path = future.result()
            if out_path:
                print(f"Saved {out_path}")
        except Exception as e:
            print(f"Error processing year {year}: {e}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Already exists: ./embeddings_1789.pkl, skipping.
Already exists: ./embeddings_1790.pkl, skipping.
Already exists: ./embeddings_1792.pkl, skipping.
Already exists: ./embeddings_1816.pkl, skipping.
Already exists: ./embeddings_1824.pkl, skipping.
Already exists: ./embeddings_1828.pkl, skipping.
Already exists: ./embeddings_1846.pkl, skipping.
Already exists: ./embeddings_1861.pkl, skipping.
Already exists: ./embeddings_1883.pkl, skipping.
Already exists: ./embeddings_1894.pkl, skipping.
Already exists: ./embeddings_1897.pkl, skipping.
Already exists: ./embeddings_1913.pkl, skipping.
Already exists: ./embeddings_1922.pkl, skipping.


Processing years:   1%|          | 1/83 [34:51<47:38:01, 2091.24s/it]

Saved ./embeddings_1939.pkl


Processing years:   2%|▏         | 2/83 [35:14<19:40:46, 874.65s/it] 

...

Saved ./embeddings_2023.pkl


Processing years:  99%|█████████▉| 82/83 [2:55:34<01:18, 78.34s/it]

Saved ./embeddings_2020.pkl


Processing years: 100%|██████████| 83/83 [2:56:27<00:00, 127.56s/it]

Saved ./embeddings_2012.pkl
